## 2. Wczytanie danych
 
 Notebook expects `train.csv` and `test.csv` in `data/raw/`. In Google Colab, you can also upload both files next to the notebook and the fallback path will still work.


## Plan pracy i odpowiedzialność

| Zakres | Miejsce w notebooku |
|---|---|
| EDA, opis danych, braki, outliery, problemy jakości danych | Sekcje 1-15 |
| Czyszczenie danych, feature engineering, pipeline, modele bazowe | Sekcje 16-19 |
| Strojenie modeli, końcowa ewaluacja, wyjaśnialność, wnioski | Sekcje 20-26 |

Po uruchomieniu wykresów warto dopisać pod najważniejszymi z nich krótkie interpretacje. W raporcie liczy się nie tylko wykres, ale też decyzja: co z niego wynika dla dalszego modelowania.


## 1. Import bibliotek i ustawienia


In [ ]:
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42


## 2. Wczytanie danych

Notebook zakłada, że pliki `train.csv` i `test.csv` znajdują się w bieżącym katalogu roboczym. W Google Colabie wystarczy wrzucić oba pliki do panelu plików po lewej stronie albo ustawić poniżej własne ścieżki do plików.


In [ ]:
from pathlib import Path

# Put the dataset files in data/raw/. If you run this notebook in Colab, you can also upload them next to the notebook.
data_dir = Path("data/raw")
train_path = data_dir / "train.csv"
test_path = data_dir / "test.csv"

if not train_path.exists():
    train_path = Path("train.csv")
if not test_path.exists():
    test_path = Path("test.csv")

raw_train = pd.read_csv(train_path)
raw_test = pd.read_csv(test_path)

print("Loaded train:", train_path, raw_train.shape)
print("Loaded test:", test_path, raw_test.shape)


## 3. Porządkowanie nazw kolumn i definicja grup zmiennych


In [ ]:
def clean_column_name(name):
    name = re.sub(r"[^0-9a-zA-Z]+", "_", str(name).strip())
    return name.strip("_").lower()


train = raw_train.copy()
test = raw_test.copy()
train.columns = [clean_column_name(col) for col in train.columns]
test.columns = [clean_column_name(col) for col in test.columns]

TARGET = "satisfaction"
TARGET_BINARY = "satisfaction_binary"
TARGET_MAP = {"neutral or dissatisfied": 0, "satisfied": 1}

for df in [train, test]:
    if TARGET in df.columns:
        df[TARGET_BINARY] = df[TARGET].map(TARGET_MAP)

id_cols = [col for col in ["unnamed_0", "id"] if col in train.columns]
categorical_cols = [col for col in ["gender", "customer_type", "type_of_travel", "class"] if col in train.columns]
continuous_cols = [col for col in ["age", "flight_distance", "departure_delay_in_minutes", "arrival_delay_in_minutes"] if col in train.columns]

non_rating_cols = set(id_cols + categorical_cols + continuous_cols + [TARGET, TARGET_BINARY])
rating_cols = [col for col in train.select_dtypes(include="number").columns if col not in non_rating_cols]

feature_cols = [col for col in train.columns if col not in id_cols + [TARGET, TARGET_BINARY]]

print("Identyfikatory:", id_cols)
print("Zmienne kategoryczne:", categorical_cols)
print("Zmienne ciągłe:", continuous_cols)
print("Oceny usług 0-5:", rating_cols)
print("Liczba potencjalnych cech:", len(feature_cols))


## 4. Pierwszy podgląd danych


In [ ]:
display(train.head())
display(test.head())

profile = pd.DataFrame({
    "dtype_train": train.dtypes.astype(str),
    "missing_train": train.isna().sum(),
    "missing_train_%": train.isna().mean() * 100,
    "unique_train": train.nunique(dropna=False),
})

if set(train.columns) == set(test.columns):
    profile["dtype_test"] = test[train.columns].dtypes.astype(str)
    profile["missing_test"] = test[train.columns].isna().sum()
    profile["missing_test_%"] = test[train.columns].isna().mean() * 100

profile = profile.sort_values(["missing_train", "unique_train"], ascending=[False, True])
display(profile)

print("Duplikaty pełnych wierszy w train:", train.duplicated().sum())
if "id" in train.columns:
    print("Duplikaty id w train:", train["id"].duplicated().sum())

print("\nStatystyki zmiennych liczbowych:")
display(train[continuous_cols + rating_cols].describe().T)

print("\nStatystyki zmiennych kategorycznych:")
display(train[categorical_cols + [TARGET]].describe().T)


### Wstępne obserwacje po profilu danych

- `unnamed_0` i `id` są identyfikatorami technicznymi, więc nie powinny być używane jako predyktory.
- W tym zbiorze `test.csv` również zawiera zmienną celu, więc można go użyć jako końcowy zbiór testowy.
- Najważniejszy widoczny brak danych dotyczy `arrival_delay_in_minutes`.
- Oceny usług są w skali 0-5. Warto sprawdzić, czy `0` oznacza realną ocenę, brak usługi czy kod specjalny.


## 5. Analiza braków danych


In [ ]:
missing = pd.DataFrame({
    "missing_train": train.isna().sum(),
    "missing_train_%": train.isna().mean() * 100,
    "missing_test": test.isna().sum(),
    "missing_test_%": test.isna().mean() * 100,
}).sort_values("missing_train", ascending=False)
display(missing)

missing_nonzero = missing[(missing["missing_train"] > 0) | (missing["missing_test"] > 0)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if not missing_nonzero.empty:
    missing_nonzero[["missing_train_%", "missing_test_%"]].plot(kind="bar", ax=axes[0])
    axes[0].set_title("Odsetek braków danych")
    axes[0].set_ylabel("% braków")
    axes[0].tick_params(axis="x", rotation=35)
else:
    axes[0].text(0.5, 0.5, "Brak brakujących wartości", ha="center", va="center")
    axes[0].set_axis_off()

sns.heatmap(train.isna(), cbar=False, yticklabels=False, ax=axes[1], cmap="viridis")
axes[1].set_title("Mapa braków danych - train")
plt.tight_layout()
plt.show()


### Komentarz do braków po połączeniu train/test

W obu plikach braki dotyczą praktycznie tylko `arrival_delay_in_minutes`. Jeżeli zsumujemy train i test, liczba braków wynosi 393, więc ich rozłożenie między zbiory wygląda proporcjonalnie do wielkości podziału danych. Nie traktujemy tego jako osobnego problemu po stronie podziału train/test.


In [ ]:
combined_missing = pd.DataFrame({
    "missing_train": train.isna().sum(),
    "missing_test": test.isna().sum(),
})
combined_missing["missing_total"] = combined_missing["missing_train"] + combined_missing["missing_test"]
combined_missing["missing_total_%"] = combined_missing["missing_total"] / (len(train) + len(test)) * 100
combined_missing = combined_missing.sort_values("missing_total", ascending=False)

display(combined_missing[combined_missing["missing_total"] > 0])
print("Łączna liczba braków w arrival_delay_in_minutes:", int(combined_missing.loc["arrival_delay_in_minutes", "missing_total"]))


## 6. Rozkład zmiennej celu


In [ ]:
target_counts = train[TARGET].value_counts()
target_share = train[TARGET].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({"count": target_counts, "share_%": target_share}))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=train, x=TARGET, order=target_counts.index, ax=axes[0])
axes[0].set_title("Liczba obserwacji w klasach")
axes[0].set_xlabel("")
axes[0].set_ylabel("Liczba pasażerów")
for container in axes[0].containers:
    axes[0].bar_label(container)

target_counts.plot(kind="pie", autopct="%1.1f%%", startangle=90, ax=axes[1])
axes[1].set_title("Udział klas")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()


## 7. Zmienne kategoryczne a satysfakcja


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for ax, col in zip(axes, categorical_cols):
    order = train[col].value_counts().index
    sns.countplot(data=train, x=col, hue=TARGET, order=order, ax=ax)
    ax.set_title(f"{col} - liczebność wg satysfakcji")
    ax.set_xlabel("")
    ax.set_ylabel("Liczba pasażerów")
    ax.tick_params(axis="x", rotation=20)

for ax in axes[len(categorical_cols):]:
    ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for ax, col in zip(axes, categorical_cols):
    tab = pd.crosstab(train[col], train[TARGET], normalize="index").mul(100)
    tab = tab.reindex(train[col].value_counts().index)
    tab.plot(kind="bar", stacked=True, ax=ax, colormap="Set2")
    ax.set_title(f"{col} - procent klas satysfakcji")
    ax.set_xlabel("")
    ax.set_ylabel("% pasażerów")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(title=TARGET, loc="upper right")

for ax in axes[len(categorical_cols):]:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

for col in categorical_cols:
    sat_rate = (
        train.groupby(col)[TARGET_BINARY]
        .agg(count="size", satisfied_rate="mean")
        .assign(satisfied_rate_pct=lambda x: x["satisfied_rate"] * 100)
        .sort_values("satisfied_rate_pct", ascending=False)
    )
    print(f"\n{col}")
    display(sat_rate[["count", "satisfied_rate_pct"]])


## 8. Zmienne liczbowe i rozkłady


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

for ax, col in zip(axes, continuous_cols):
    plot_df = train[[col, TARGET]].dropna().copy()
    title_suffix = ""
    if "delay" in col:
        plot_df[col] = plot_df[col].clip(upper=plot_df[col].quantile(0.99))
        title_suffix = "(przycięte do 99. percentyla)"
    sns.histplot(data=plot_df, x=col, hue=TARGET, kde=True, bins=40, stat="density", common_norm=False, ax=ax)
    ax.set_title(f"Rozkład: {col} {title_suffix}")
    ax.set_xlabel(col)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

for ax, col in zip(axes, continuous_cols):
    plot_df = train[[col, TARGET]].dropna().copy()
    if "delay" in col:
        plot_df[col] = plot_df[col].clip(upper=plot_df[col].quantile(0.99))
    sns.boxplot(data=plot_df, x=TARGET, y=col, showfliers=False, ax=ax)
    ax.set_title(f"{col} wg satysfakcji")
    ax.set_xlabel("")

plt.tight_layout()
plt.show()


## 9. Oceny usług 0-5


In [ ]:
zero_ratings = train[rating_cols].eq(0).sum().sort_values(ascending=False)
zero_ratings_pct = train[rating_cols].eq(0).mean().mul(100).sort_values(ascending=False)
zero_table = pd.DataFrame({"zero_count": zero_ratings, "zero_%": zero_ratings_pct})
display(zero_table)

plt.figure(figsize=(11, 6))
zero_table["zero_%"].sort_values().plot(kind="barh", color="#66c2a5")
plt.title("Odsetek wartości 0 w ocenach usług")
plt.xlabel("% obserwacji z oceną 0")
plt.ylabel("")
plt.tight_layout()
plt.show()


### Interpretacja wartości 0 w ocenach

Komentarze do zbioru sugerują, że zmienne ocenowe są w skali 1-5, a `0` oznacza brak odpowiedzi. To ważne, bo model porządkowy mógłby potraktować `0` jako ocenę gorszą od `1`. Dlatego w części modelowej zachowujemy informację o braku odpowiedzi w osobnych flagach, a same zera w ocenach zamieniamy na wartości brakujące.


In [ ]:
zero_response_rows = []
for col in rating_cols:
    zero_mask = train[col].eq(0)
    answered_mask = ~zero_mask
    zero_response_rows.append({
        "feature": col,
        "zero_count": int(zero_mask.sum()),
        "zero_%": zero_mask.mean() * 100,
        "satisfied_rate_zero_%": train.loc[zero_mask, TARGET_BINARY].mean() * 100 if zero_mask.any() else np.nan,
        "satisfied_rate_answered_%": train.loc[answered_mask, TARGET_BINARY].mean() * 100 if answered_mask.any() else np.nan,
    })

zero_response_df = pd.DataFrame(zero_response_rows)
zero_response_df["difference_zero_minus_answered_pp"] = (
    zero_response_df["satisfied_rate_zero_%"] - zero_response_df["satisfied_rate_answered_%"]
)
zero_response_df = zero_response_df.sort_values("zero_%", ascending=False)
display(zero_response_df)

plot_zero_df = zero_response_df[zero_response_df["zero_count"] > 0].copy()
if not plot_zero_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.barplot(data=plot_zero_df, y="feature", x="zero_%", ax=axes[0], color="#66c2a5")
    axes[0].set_title("Częstość braku odpowiedzi (wartość 0)")
    axes[0].set_xlabel("% wartości 0")
    axes[0].set_ylabel("")

    sns.barplot(
        data=plot_zero_df,
        y="feature",
        x="difference_zero_minus_answered_pp",
        ax=axes[1],
        color="#fc8d62",
    )
    axes[1].axvline(0, color="black", linewidth=1)
    axes[1].set_title("Różnica % zadowolonych: brak odpowiedzi vs odpowiedź")
    axes[1].set_xlabel("punkty procentowe")
    axes[1].set_ylabel("")
    plt.tight_layout()
    plt.show()


In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(rating_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4.2 * n_rows))
axes = axes.ravel()

for ax, col in zip(axes, rating_cols):
    tab = pd.crosstab(train[col], train[TARGET], normalize="index").mul(100)
    tab.plot(kind="bar", stacked=True, ax=ax, colormap="Set2", width=0.85)
    ax.set_title(col)
    ax.set_xlabel("Ocena")
    ax.set_ylabel("% pasażerów")
    ax.legend(title=TARGET, fontsize=8)

for ax in axes[len(rating_cols):]:
    ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
rating_means = train.groupby(TARGET)[rating_cols].mean().T
rating_means["diff_satisfied_minus_neutral"] = rating_means["satisfied"] - rating_means["neutral or dissatisfied"]
rating_means = rating_means.sort_values("diff_satisfied_minus_neutral", ascending=False)
display(rating_means)

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
sns.heatmap(
    rating_means[["neutral or dissatisfied", "satisfied"]],
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=1,
    vmax=5,
    ax=axes[0],
)
axes[0].set_title("Średnia ocena usług wg klasy satysfakcji")
axes[0].set_xlabel("")
axes[0].set_ylabel("")

rating_means["diff_satisfied_minus_neutral"].sort_values().plot(kind="barh", ax=axes[1], color="#8da0cb")
axes[1].set_title("Różnica średnich: satisfied - neutral/dissatisfied")
axes[1].set_xlabel("Różnica średniej oceny")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()


## 10. Korelacje i relacje między cechami


In [ ]:
corr_cols = continuous_cols + rating_cols + [TARGET_BINARY]
corr = train[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(15, 12))
sns.heatmap(corr, cmap="vlag", center=0, linewidths=0.3, cbar_kws={"shrink": 0.8})
plt.title("Macierz korelacji zmiennych liczbowych")
plt.tight_layout()
plt.show()

target_corr = corr[TARGET_BINARY].drop(TARGET_BINARY).sort_values()
display(target_corr.to_frame("correlation_with_satisfaction"))

plt.figure(figsize=(10, 8))
colors = ["#fc8d62" if value < 0 else "#66c2a5" for value in target_corr]
target_corr.plot(kind="barh", color=colors)
plt.title("Korelacja cech liczbowych ze zmienną celu")
plt.xlabel("Korelacja z satisfaction_binary")
plt.ylabel("")
plt.axvline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()


In [ ]:
selected_for_pairplot = [
    "online_boarding",
    "inflight_entertainment",
    "seat_comfort",
    "flight_distance",
    "age",
    TARGET,
]
selected_for_pairplot = [col for col in selected_for_pairplot if col in train.columns]

pair_sample = train[selected_for_pairplot].dropna().sample(n=min(4000, len(train)), random_state=RANDOM_STATE)
sns.pairplot(pair_sample, hue=TARGET, diag_kind="hist", corner=True, plot_kws={"alpha": 0.35, "s": 12})
plt.suptitle("Pairplot wybranych cech", y=1.02)
plt.show()


## 11. Szczegółowa analiza opóźnień


In [ ]:
delay_cols = [col for col in ["departure_delay_in_minutes", "arrival_delay_in_minutes"] if col in train.columns]

if len(delay_cols) == 2:
    delay_sample = train[delay_cols + [TARGET]].dropna().sample(n=min(12000, len(train)), random_state=RANDOM_STATE).copy()
    for col in delay_cols:
        delay_sample[col] = delay_sample[col].clip(upper=train[col].quantile(0.99))

    plt.figure(figsize=(9, 7))
    sns.scatterplot(
        data=delay_sample,
        x="departure_delay_in_minutes",
        y="arrival_delay_in_minutes",
        hue=TARGET,
        alpha=0.35,
        s=18,
    )
    plt.title("Opóźnienie odlotu vs opóźnienie przylotu")
    plt.tight_layout()
    plt.show()

    display(train[delay_cols].corr())

    segment_df = train.copy()
    delay_bins = [-1, 0, 15, 60, 180, np.inf]
    delay_labels = ["0", "1-15", "16-60", "61-180", "180+"]
    segment_df["arrival_delay_bin"] = pd.cut(segment_df["arrival_delay_in_minutes"], bins=delay_bins, labels=delay_labels)

    delay_sat = (
        segment_df.groupby("arrival_delay_bin", observed=False)[TARGET_BINARY]
        .agg(count="size", satisfied_rate="mean")
        .assign(satisfied_rate_pct=lambda x: x["satisfied_rate"] * 100)
    )
    display(delay_sat[["count", "satisfied_rate_pct"]])

    plt.figure(figsize=(9, 5))
    sns.barplot(data=delay_sat.reset_index(), x="arrival_delay_bin", y="satisfied_rate_pct", color="#8da0cb")
    plt.title("Odsetek zadowolonych wg opóźnienia przylotu")
    plt.xlabel("Opóźnienie przylotu [min]")
    plt.ylabel("% zadowolonych")
    plt.tight_layout()
    plt.show()


## 12. Outliery i nietypowe wartości


In [ ]:
def iqr_outlier_summary(df, columns):
    rows = []
    for col in columns:
        series = df[col].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((series < lower) | (series > upper)).sum()
        rows.append({
            "column": col,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": outliers,
            "outlier_%": outliers / len(series) * 100,
            "min": series.min(),
            "max": series.max(),
        })
    return pd.DataFrame(rows).sort_values("outlier_%", ascending=False)


outlier_table = iqr_outlier_summary(train, continuous_cols)
display(outlier_table)

fig, axes = plt.subplots(1, len(continuous_cols), figsize=(4.5 * len(continuous_cols), 5))
if len(continuous_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, continuous_cols):
    plot_series = train[col].dropna().copy()
    if "delay" in col:
        plot_series = plot_series.clip(upper=plot_series.quantile(0.99))
    sns.boxplot(y=plot_series, ax=ax, color="#a6d854")
    ax.set_title(col)
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


## 13. Porównanie zbioru treningowego i testowego


In [ ]:
compare_numeric = continuous_cols + rating_cols
train_means = train[compare_numeric].mean(numeric_only=True)
test_means = test[compare_numeric].mean(numeric_only=True)
compare_table = pd.DataFrame({
    "train_mean": train_means,
    "test_mean": test_means,
    "abs_diff": (train_means - test_means).abs(),
    "relative_diff_%": ((train_means - test_means).abs() / train_means.replace(0, np.nan) * 100),
}).sort_values("relative_diff_%", ascending=False)
display(compare_table)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for ax, col in zip(axes, continuous_cols):
    sns.kdeplot(train[col].dropna(), label="train", ax=ax)
    sns.kdeplot(test[col].dropna(), label="test", ax=ax)
    ax.set_title(f"Train vs test: {col}")
    ax.legend()

plt.tight_layout()
plt.show()

if TARGET in test.columns:
    target_compare = pd.concat(
        [
            train[TARGET].value_counts(normalize=True).rename("train"),
            test[TARGET].value_counts(normalize=True).rename("test"),
        ],
        axis=1,
    ).mul(100)
    display(target_compare)

    target_compare.plot(kind="bar", figsize=(8, 5), color=["#66c2a5", "#fc8d62"])
    plt.title("Porównanie rozkładu zmiennej celu: train vs test")
    plt.ylabel("% obserwacji")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()


## 14. Segmenty pasażerów


In [ ]:
segment_df = train.copy()

segment_df["age_group"] = pd.cut(
    segment_df["age"],
    bins=[0, 18, 25, 35, 45, 55, 65, 100],
    labels=["0-18", "19-25", "26-35", "36-45", "46-55", "56-65", "66+"],
)

segment_df["distance_group"] = pd.qcut(segment_df["flight_distance"], q=5, duplicates="drop")
segment_cols = ["age_group", "distance_group", "class", "type_of_travel", "customer_type"]

fig, axes = plt.subplots(len(segment_cols), 1, figsize=(12, 4 * len(segment_cols)))

for ax, col in zip(axes, segment_cols):
    sat_rate = (
        segment_df.groupby(col, observed=False)[TARGET_BINARY]
        .agg(count="size", satisfied_rate="mean")
        .assign(satisfied_rate_pct=lambda x: x["satisfied_rate"] * 100)
        .reset_index()
    )
    display(sat_rate[[col, "count", "satisfied_rate_pct"]])
    sns.barplot(data=sat_rate, x=col, y="satisfied_rate_pct", ax=ax, color="#8da0cb")
    ax.set_title(f"% zadowolonych pasażerów wg: {col}")
    ax.set_ylabel("% zadowolonych")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## 15. Podsumowanie EDA - wnioski startowe

Najważniejsze obserwacje po eksploracji danych:

1. Zmienna celu jest umiarkowanie niezbalansowana: w `train.csv` około 56,7% to `neutral or dissatisfied`, a około 43,3% to `satisfied`.
2. Braki danych występują prawie wyłącznie w `arrival_delay_in_minutes`. Łącznie w train i test jest ich 393, więc skala problemu jest mała.
3. `id` i `unnamed_0` są identyfikatorami technicznymi i należy je usunąć z predyktorów.
4. Najsilniej z satysfakcją wstępnie korelują m.in. `online_boarding`, `inflight_entertainment`, `seat_comfort`, `on_board_service`, `leg_room_service` oraz `flight_distance`.
5. Zmienne opóźnień i dystansu lotu są skośne oraz zawierają obserwacje odstające, więc dodajemy transformacje `log1p` i wskaźniki opóźnienia.
6. W ocenach usług pojawiają się wartości `0`. Przyjmujemy roboczo, że oznaczają brak odpowiedzi, a nie najgorszą ocenę. W modelowaniu dodajemy dla nich flagi i zamieniamy `0` na brak danych.
7. Dane można podzielić na trzy grupy: obiektywne informacje o pasażerze/locie, zmienne kontekstowe o nie do końca jasnej definicji oraz samoopisowe oceny usług. Dlatego warto porównać model pełny z modelami używającymi tylko wybranych grup cech.


# Część modelowa - szkielet dalszej pracy

Od tego miejsca notebook zawiera strukturę pod kolejne etapy projektu. Komórki są przygotowane tak, żeby dało się je rozwijać bez przebudowy całego pliku.


## 16. Decyzje o czyszczeniu danych i feature engineering

Przyjęte decyzje:

- Usuwamy `id` i `unnamed_0` z cech modelu.
- Uzupełniamy braki liczbowe medianą w pipeline.
- Zmienne kategoryczne kodujemy przez one-hot encoding.
- Zmienne liczbowe skalujemy, żeby modele liniowe miały stabilniejsze współczynniki.
- Dla opóźnień i dystansu dodajemy transformacje `log1p`.
- Dla brakującego `arrival_delay_in_minutes` dodajemy flagę `arrival_delay_missing`.
- Dla ocen usług traktujemy `0` jako brak odpowiedzi: tworzymy flagi `*_no_answer`, a same zera w kolumnach ocen zamieniamy na `NaN`.
- Tworzymy cechy zbiorcze dla ocen usług, np. średnią ocenę, minimum, liczbę niskich ocen oraz liczbę braków odpowiedzi.


In [ ]:
def add_features(df):
    df = df.copy()

    if "arrival_delay_in_minutes" in df.columns:
        df["arrival_delay_missing"] = df["arrival_delay_in_minutes"].isna().astype(int)

    if {"departure_delay_in_minutes", "arrival_delay_in_minutes"}.issubset(df.columns):
        delay_cols_local = ["departure_delay_in_minutes", "arrival_delay_in_minutes"]
        df["total_delay_minutes"] = df[delay_cols_local].sum(axis=1, min_count=2)
        df["arrival_departure_delay_diff"] = df["arrival_delay_in_minutes"] - df["departure_delay_in_minutes"]
        df["had_departure_delay"] = (df["departure_delay_in_minutes"].fillna(0) > 0).astype(int)
        df["had_arrival_delay"] = (df["arrival_delay_in_minutes"].fillna(0) > 0).astype(int)
        df["log_departure_delay"] = np.log1p(df["departure_delay_in_minutes"].clip(lower=0))
        df["log_arrival_delay"] = np.log1p(df["arrival_delay_in_minutes"].clip(lower=0))

    if "flight_distance" in df.columns:
        df["log_flight_distance"] = np.log1p(df["flight_distance"].clip(lower=0))

    zeroreplacer = np.nan

    available_rating_cols = [col for col in rating_cols if col in df.columns]
    if available_rating_cols:
        for col in available_rating_cols:
            df[f"{col}_no_answer"] = df[col].eq(0).astype(int)

        ratings_without_zero = df[available_rating_cols].replace(0, zeroreplacer)
        df[available_rating_cols] = ratings_without_zero

        df["service_rating_mean"] = ratings_without_zero.mean(axis=1)
        df["service_rating_min"] = ratings_without_zero.min(axis=1)
        df["service_rating_max"] = ratings_without_zero.max(axis=1)
        df["service_rating_std"] = ratings_without_zero.std(axis=1)
        df["rating_answered_count"] = ratings_without_zero.notna().sum(axis=1)
        df["rating_no_answer_count"] = ratings_without_zero.isna().sum(axis=1)
        df["rating_no_answer_share"] = df["rating_no_answer_count"] / len(available_rating_cols)
        df["low_service_rating_count"] = ratings_without_zero.le(2).sum(axis=1)
        df["high_service_rating_count"] = ratings_without_zero.ge(4).sum(axis=1)

    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[0, 18, 25, 35, 45, 55, 65, 100],
            labels=["0-18", "19-25", "26-35", "36-45", "46-55", "56-65", "66+"],
        ).astype("object")

    return df


train_fe = add_features(train)
test_fe = add_features(test)

model_drop_cols = id_cols + [TARGET, TARGET_BINARY]
X_train_full = train_fe.drop(columns=[col for col in model_drop_cols if col in train_fe.columns])
y_train_full = train_fe[TARGET_BINARY]

X_test_final = test_fe.drop(columns=[col for col in model_drop_cols if col in test_fe.columns])
y_test_final = test_fe[TARGET_BINARY] if TARGET_BINARY in test_fe.columns else None

print("X_train_full:", X_train_full.shape)
print("y_train_full:", y_train_full.shape)
print("X_test_final:", X_test_final.shape)
print("Liczba dodanych flag braku odpowiedzi:", len([c for c in X_train_full.columns if c.endswith("_no_answer")]))


## 17. Podział danych i pipeline przetwarzania


In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocess(X):
    numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

    transformers = []
    if numeric_cols:
        transformers.append((
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ))
    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_onehot_encoder()),
            ]),
            categorical_cols,
        ))
    return ColumnTransformer(transformers=transformers)


numeric_cols_model = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols_model = X_train.select_dtypes(exclude=np.number).columns.tolist()
preprocess = make_preprocess(X_train)

print("Liczba cech liczbowych:", len(numeric_cols_model))
print("Liczba cech kategorycznych:", len(categorical_cols_model))


## 18. Modele bazowe


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

models = {
    "Dummy majority": DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),
    "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Gaussian Naive Bayes": GaussianNB(),
    "Decision tree": DecisionTreeClassifier(max_depth=18, min_samples_leaf=20, random_state=RANDOM_STATE, class_weight="balanced"),
    "Random forest": RandomForestClassifier(
        n_estimators=220,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        class_weight="balanced_subsample",
    ),
    "Extra trees": ExtraTreesClassifier(
        n_estimators=220,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "Hist gradient boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}


def get_positive_scores(pipe, X):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]
    if hasattr(pipe, "decision_function"):
        return pipe.decision_function(X)
    return pipe.predict(X)


def metric_row(name, y_true, pred, scores):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, scores) if scores is not None else np.nan,
    }


def evaluate_model(name, model, X_train, y_train, X_valid, y_valid):
    pipe = Pipeline(steps=[
        ("preprocess", make_preprocess(X_train)),
        ("model", clone(model)),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_valid)
    scores = get_positive_scores(pipe, X_valid)
    return pipe, metric_row(name, y_valid, pred, scores)


trained_models = {}
results = []

for name, model in models.items():
    print(f"Trenuję: {name}")
    fitted_pipe, metrics = evaluate_model(name, model, X_train, y_train, X_valid, y_valid)
    trained_models[name] = fitted_pipe
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)

plt.figure(figsize=(11, 6))
sns.barplot(data=results_df, y="model", x="f1", color="#66c2a5")
plt.title("Porównanie modeli bazowych - F1 na walidacji")
plt.xlabel("F1")
plt.ylabel("")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_pipe = trained_models[best_model_name]

print("Najlepszy model wg F1:", best_model_name)
y_valid_pred = best_pipe.predict(X_valid)
y_valid_score = get_positive_scores(best_pipe, X_valid)

print(classification_report(
    y_valid,
    y_valid_pred,
    target_names=["neutral/dissatisfied", "satisfied"],
    zero_division=0,
))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(confusion_matrix(y_valid, y_valid_pred), annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Macierz pomyłek - walidacja")
axes[0].set_xlabel("Predykcja")
axes[0].set_ylabel("Rzeczywistość")

RocCurveDisplay.from_predictions(y_valid, y_valid_score, ax=axes[1])
axes[1].set_title("Krzywa ROC - walidacja")

PrecisionRecallDisplay.from_predictions(y_valid, y_valid_score, ax=axes[2])
axes[2].set_title("Krzywa Precision-Recall - walidacja")

plt.tight_layout()
plt.show()


## 19. Eksperyment: modele według grup cech

Ten eksperyment realizuje pomysł porównania modeli używających różnych typów informacji:

- **Dane obiektywne:** płeć, wiek, klasa biletu, dystans lotu i opóźnienia. To informacje, które linia lotnicza powinna znać bez ankiety.
- **Kontekst klienta/podróży:** `type_of_travel` i `customer_type`. Są przydatne, ale ich interpretacja jest mniej przejrzysta, szczególnie dla `customer_type`.
- **Dane samoopisowe:** oceny usług 1-5 oraz informacja, że pasażer nie odpowiedział (`0` w oryginalnych danych).

Model tylko na danych obiektywnych prawdopodobnie będzie słabszy od pełnego modelu. To nie jest błąd, tylko ważny wniosek: bardzo duża część sygnału predykcyjnego pochodzi z ankietowych ocen usług.


In [ ]:
objective_base_cols = [
    "gender",
    "age",
    "class",
    "flight_distance",
    "departure_delay_in_minutes",
    "arrival_delay_in_minutes",
]
objective_derived_cols = [
    "arrival_delay_missing",
    "total_delay_minutes",
    "arrival_departure_delay_diff",
    "had_departure_delay",
    "had_arrival_delay",
    "log_departure_delay",
    "log_arrival_delay",
    "log_flight_distance",
    "age_group",
]
travel_context_cols = ["type_of_travel", "customer_type"]
rating_indicator_cols = [f"{col}_no_answer" for col in rating_cols]
rating_summary_cols = [
    "service_rating_mean",
    "service_rating_min",
    "service_rating_max",
    "service_rating_std",
    "rating_answered_count",
    "rating_no_answer_count",
    "rating_no_answer_share",
    "low_service_rating_count",
    "high_service_rating_count",
]

objective_cols = [col for col in objective_base_cols + objective_derived_cols if col in X_train_full.columns]
objective_plus_context_cols = [col for col in objective_cols + travel_context_cols if col in X_train_full.columns]
self_reported_cols = [col for col in rating_cols + rating_indicator_cols + rating_summary_cols if col in X_train_full.columns]
full_cols = X_train_full.columns.tolist()

feature_sets = {
    "objective_only": objective_cols,
    "objective_plus_context": objective_plus_context_cols,
    "self_reported_ratings_only": self_reported_cols,
    "all_features": full_cols,
}

feature_set_summary = pd.DataFrame([
    {"feature_set": name, "n_features_before_encoding": len(cols), "features": ", ".join(cols)}
    for name, cols in feature_sets.items()
])
display(feature_set_summary[["feature_set", "n_features_before_encoding"]])

feature_set_models = {
    "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Hist gradient boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

feature_set_rows = []
feature_set_pipes = {}

for feature_set_name, cols in feature_sets.items():
    X_train_fs = X_train_full.loc[X_train.index, cols]
    X_valid_fs = X_train_full.loc[X_valid.index, cols]

    for model_name, model in feature_set_models.items():
        pipe = Pipeline(steps=[
            ("preprocess", make_preprocess(X_train_fs)),
            ("model", clone(model)),
        ])
        pipe.fit(X_train_fs, y_train)
        pred = pipe.predict(X_valid_fs)
        scores = get_positive_scores(pipe, X_valid_fs)
        row = metric_row(f"{model_name} | {feature_set_name}", y_valid, pred, scores)
        row["feature_set"] = feature_set_name
        row["base_model"] = model_name
        row["n_features_before_encoding"] = len(cols)
        feature_set_rows.append(row)
        feature_set_pipes[(feature_set_name, model_name)] = pipe

feature_set_results = pd.DataFrame(feature_set_rows).sort_values(["f1", "roc_auc"], ascending=False)
display(feature_set_results)

plt.figure(figsize=(12, 7))
sns.barplot(data=feature_set_results, y="model", x="f1", color="#8da0cb")
plt.title("Wpływ grup cech na jakość modelu - F1 na walidacji")
plt.xlabel("F1")
plt.ylabel("")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()


## 20. Walidacja krzyżowa


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

# Walidację krzyżową robimy dla najważniejszych modeli, żeby czas wykonania w Colabie był rozsądny.
cv_model_names = [
    "Dummy majority",
    "Logistic regression",
    "Random forest",
    "Extra trees",
    "Hist gradient boosting",
]

cv_rows = []
for name in cv_model_names:
    model = models[name]
    pipe = Pipeline(steps=[
        ("preprocess", make_preprocess(X_train_full)),
        ("model", clone(model)),
    ])
    cv_scores = cross_validate(
        pipe,
        X_train_full,
        y_train_full,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
    )
    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = cv_scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = cv_scores[f"test_{metric}"].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values("f1_mean", ascending=False)
display(cv_results)


## 21. Krzywe uczenia modeli

Ta sekcja pokazuje, jak modele zachowują się przy rosnącej liczbie przykładów treningowych. Dla każdego modelu trenujemy pipeline na 10%, 25%, 50%, 75% i 100% części treningowej, a następnie zapisujemy F1 na treningu, F1 na walidacji oraz czas dopasowania. Dzięki temu widać, czy model realnie zyskuje na większej ilości danych i jaki jest koszt obliczeniowy tego zysku.

Wykresy są zapisywane do katalogu `raport_assets`, żeby można było bezpośrednio wykorzystać je w raporcie Typst.


In [ ]:
from pathlib import Path
from time import perf_counter
import re

RUN_LEARNING_CURVES = True
learning_fractions = [0.10, 0.25, 0.50, 0.75, 1.00]
learning_asset_dir = Path("raport_assets")
learning_asset_dir.mkdir(exist_ok=True)


def stratified_fraction_index(y, frac):
    if frac >= 1:
        return y.index
    return (
        y.groupby(y, group_keys=False)
        .sample(frac=frac, random_state=RANDOM_STATE)
        .index
    )


def slugify(value):
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


if RUN_LEARNING_CURVES:
    learning_rows = []

    for model_name, model in models.items():
        print(f"Krzywa uczenia: {model_name}")

        for frac in learning_fractions:
            subset_idx = stratified_fraction_index(y_train, frac)
            X_part = X_train.loc[subset_idx]
            y_part = y_train.loc[subset_idx]

            pipe = Pipeline(steps=[
                ("preprocess", make_preprocess(X_part)),
                ("model", clone(model)),
            ])

            start = perf_counter()
            pipe.fit(X_part, y_part)
            fit_time = perf_counter() - start

            train_pred = pipe.predict(X_part)
            valid_pred = pipe.predict(X_valid)

            learning_rows.append({
                "model": model_name,
                "fraction": frac,
                "train_size": len(X_part),
                "train_f1": f1_score(y_part, train_pred, zero_division=0),
                "valid_f1": f1_score(y_valid, valid_pred, zero_division=0),
                "fit_time_seconds": fit_time,
            })

    learning_curves_df = pd.DataFrame(learning_rows)
    learning_curves_df.to_csv(learning_asset_dir / "learning_curves_results.csv", index=False)
else:
    csv_path = learning_asset_dir / "learning_curves_results.csv"
    if csv_path.exists():
        learning_curves_df = pd.read_csv(csv_path)
    else:
        learning_curves_df = pd.DataFrame()
        print("Brak zapisanych wyników krzywych uczenia.")

if not learning_curves_df.empty:
    display(learning_curves_df)

    for model_name, model_df in learning_curves_df.groupby("model", sort=False):
        slug = slugify(model_name)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

        axes[0].plot(model_df["train_size"], model_df["train_f1"], marker="o", linewidth=2, label="F1 trening")
        axes[0].plot(model_df["train_size"], model_df["valid_f1"], marker="o", linewidth=2, label="F1 walidacja")
        axes[0].set_title(f"Krzywa uczenia - {model_name}")
        axes[0].set_xlabel("Liczba przykładów treningowych")
        axes[0].set_ylabel("F1")
        axes[0].set_ylim(0, 1.03)
        axes[0].legend(loc="lower right")

        axes[1].plot(model_df["train_size"], model_df["fit_time_seconds"], marker="o", linewidth=2, color="#54a24b")
        axes[1].set_title("Czas trenowania")
        axes[1].set_xlabel("Liczba przykładów treningowych")
        axes[1].set_ylabel("Sekundy")

        for ax in axes:
            ax.tick_params(axis="x", rotation=20)

        plt.tight_layout()
        plt.savefig(learning_asset_dir / f"learning_curve_{slug}.png", dpi=170, bbox_inches="tight")
        plt.show()

    fig, ax = plt.subplots(figsize=(11, 6.5))
    for model_name, model_df in learning_curves_df.groupby("model", sort=False):
        ax.plot(model_df["train_size"], model_df["valid_f1"], marker="o", linewidth=2, label=model_name)
    ax.set_title("Krzywe uczenia modeli - F1 na walidacji")
    ax.set_xlabel("Liczba przykładów treningowych")
    ax.set_ylabel("F1 walidacyjne")
    ax.set_ylim(0, 1.03)
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(learning_asset_dir / "learning_curves_models.png", dpi=170, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 6.5))
    zoom_df = learning_curves_df[learning_curves_df["model"] != "Dummy majority"]
    for model_name, model_df in zoom_df.groupby("model", sort=False):
        ax.plot(model_df["train_size"], model_df["valid_f1"], marker="o", linewidth=2, label=model_name)
    ax.set_title("Krzywe uczenia modeli - F1 na walidacji (bez Dummy)")
    ax.set_xlabel("Liczba przykładów treningowych")
    ax.set_ylabel("F1 walidacyjne")
    ax.set_ylim(max(0, zoom_df["valid_f1"].min() - 0.06), 1.01)
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(learning_asset_dir / "learning_curves_models_zoom.png", dpi=170, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 6.5))
    for model_name, model_df in learning_curves_df.groupby("model", sort=False):
        ax.plot(model_df["train_size"], model_df["fit_time_seconds"], marker="o", linewidth=2, label=model_name)
    ax.set_title("Czas trenowania modeli przy rosnącej liczbie przykładów")
    ax.set_xlabel("Liczba przykładów treningowych")
    ax.set_ylabel("Czas trenowania [s]")
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(learning_asset_dir / "learning_times_models.png", dpi=170, bbox_inches="tight")
    plt.show()


## 22. Strojenie hiperparametrów


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

hgb_pipe = Pipeline(steps=[
    ("preprocess", make_preprocess(X_train_full)),
    ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
])

hgb_param_dist = {
    "model__learning_rate": [0.03, 0.05, 0.08, 0.10],
    "model__max_iter": [120, 200, 300],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [20, 50, 100],
    "model__l2_regularization": [0.0, 0.01, 0.1],
}

# Ustaw False, jeśli trzeba szybko przejść przez cały notebook.
RUN_TUNING = True

if RUN_TUNING:
    hgb_search = RandomizedSearchCV(
        estimator=hgb_pipe,
        param_distributions=hgb_param_dist,
        n_iter=8,
        scoring="f1",
        cv=3,
        n_jobs=-1,
        verbose=1,
        random_state=RANDOM_STATE,
        refit=True,
    )
    hgb_search.fit(X_train_full, y_train_full)
    tuned_model = hgb_search.best_estimator_

    print("Najlepsze parametry:", hgb_search.best_params_)
    print("Najlepszy wynik CV F1:", hgb_search.best_score_)

    tuning_results = pd.DataFrame(hgb_search.cv_results_).sort_values("rank_test_score")
    display(tuning_results[[
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_model__learning_rate",
        "param_model__max_iter",
        "param_model__max_leaf_nodes",
        "param_model__min_samples_leaf",
        "param_model__l2_regularization",
    ]].head(10))
else:
    tuned_model = None
    print("Strojenie pominięte. Finalny model użyje najlepszego modelu bazowego.")


## 23. Ewaluacja na końcowym zbiorze testowym


In [ ]:
if "tuned_model" in globals() and tuned_model is not None:
    final_model = tuned_model
    final_model_name = "Tuned Hist gradient boosting"
    print("Finalny model:", final_model_name)
else:
    final_model = clone(best_pipe)
    final_model.fit(X_train_full, y_train_full)
    final_model_name = f"Best baseline: {best_model_name}"
    print("Finalny model:", final_model_name)

if y_test_final is not None:
    y_test_pred = final_model.predict(X_test_final)
    y_test_score = get_positive_scores(final_model, X_test_final)

    final_metrics = metric_row(final_model_name, y_test_final, y_test_pred, y_test_score)
    display(pd.DataFrame([final_metrics]))
    print(classification_report(
        y_test_final,
        y_test_pred,
        target_names=["neutral/dissatisfied", "satisfied"],
        zero_division=0,
    ))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.heatmap(confusion_matrix(y_test_final, y_test_pred), annot=True, fmt="d", cmap="Blues", ax=axes[0])
    axes[0].set_title("Macierz pomyłek - test")
    axes[0].set_xlabel("Predykcja")
    axes[0].set_ylabel("Rzeczywistość")

    RocCurveDisplay.from_predictions(y_test_final, y_test_score, ax=axes[1])
    axes[1].set_title("ROC - test")

    PrecisionRecallDisplay.from_predictions(y_test_final, y_test_score, ax=axes[2])
    axes[2].set_title("Precision-Recall - test")

    plt.tight_layout()
    plt.show()
else:
    print("Brak zmiennej celu w test.csv - pomijam ewaluację końcową.")


## 24. Wyjaśnialność modelu


In [ ]:
from sklearn.inspection import permutation_importance

explain_model = final_model if "final_model" in globals() else best_pipe

if y_test_final is not None:
    X_importance_source = X_test_final
    y_importance_source = y_test_final
    importance_source_name = "test"
else:
    X_importance_source = X_valid
    y_importance_source = y_valid
    importance_source_name = "valid"

importance_sample_size = min(8000, len(X_importance_source))
X_importance = X_importance_source.sample(n=importance_sample_size, random_state=RANDOM_STATE)
y_importance = y_importance_source.loc[X_importance.index]

perm = permutation_importance(
    explain_model,
    X_importance,
    y_importance,
    n_repeats=8,
    random_state=RANDOM_STATE,
    scoring="f1",
    n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": X_importance.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

print("Permutation importance policzone na zbiorze:", importance_source_name)
display(perm_df.head(20))

plt.figure(figsize=(10, 8))
sns.barplot(data=perm_df.head(20), y="feature", x="importance_mean", color="#66c2a5")
plt.title("Permutation importance - top 20 cech")
plt.xlabel("Spadek F1 po losowym przetasowaniu cechy")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 25. Tabela wyników do raportu

Ta sekcja zbiera najważniejsze tabele wyników wygenerowane wcześniej. Po ponownym uruchomieniu notebooka można je bezpośrednio przenieść do raportu.


In [ ]:
report_tables = {}

if "results_df" in globals():
    report_tables["Modele bazowe - walidacja"] = results_df
if "feature_set_results" in globals():
    report_tables["Porównanie grup cech - walidacja"] = feature_set_results
if "cv_results" in globals():
    report_tables["Walidacja krzyżowa"] = cv_results
if "final_metrics" in globals():
    report_tables["Finalny test"] = pd.DataFrame([final_metrics])

if not report_tables:
    print("Najpierw uruchom sekcje modelowe.")
else:
    for title, table in report_tables.items():
        print("\n" + title)
        display(table)
